<h3>실습 안내</h3><p>짧은 Document 세 건을 OpenAI 임베딩으로 숫자 벡터로 바꿔 저장합니다. <code>k</code>는 검색할 문서 개수입니다. k만 1에서 2로 바꾸고 검색 결과가 어떻게 달라지는지 비교합니다.</p>
<h3>벡터스토어 호출을 단계별로 읽기</h3><p><code>OpenAIEmbeddings()</code>는 설정 객체를 만들고, 실제 문서 임베딩은 <code>add_documents()</code>를 호출할 때 실행됩니다. <code>similarity_search_with_score(question, k=2)</code>는 질문을 같은 모델로 임베딩한 뒤 <code>(Document, 코사인 유사도)</code> 두 건을 반환합니다. 이 <code>InMemoryVectorStore</code> 점수는 클수록 질문과 더 가깝습니다.</p><p>유사도는 관련성 확률이 아니므로 0.8을 80% 관련으로 해석하지 않습니다. 정확한 제품 코드가 중요한 질문은 의미 벡터만으로 부족할 수 있으며 Ch 12에서 키워드 검색과 비교합니다.</p>

In [1]:
# 임베딩과 메모리 벡터스토어에 필요한 패키지를 설치합니다.
# ==는 검증된 버전을 고정하고 -q는 설치 로그를 줄입니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


<h3>1. API 키 준비</h3><p>현재 작업 폴더에서 <code>.env</code>를 찾아 실제 임베딩 호출을 준비합니다.</p>

In [2]:
# env_file은 find_dotenv()가 찾은 .env 경로 문자열입니다.
# load_dotenv()는 파일을 로드하고 os.getenv()는 OPENAI_API_KEY 존재를 확인합니다.
import os
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, ".env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
print("API 키 준비 완료")

API 키 준비 완료


<h3>2. 라벨이 있는 문서 준비</h3><p>검색된 문서를 이름으로 구분할 수 있게 각 Document에 D01, D02, D03이라는 문서 ID(<code>doc_id</code>)를 넣습니다.</p>

In [3]:
# raw_documents의 각 튜플은 (문서 ID, 본문) 두 값을 묶습니다.
# for로 튜플을 doc_id와 content로 나누어 Document를 하나씩 추가합니다.
from langchain_core.documents import Document

raw_documents = [
    ("D01", "미르시립도서관의 정기 휴관일은 매주 월요일입니다."),
    ("D02", "해솔식당의 점심 영업은 오전 11시에 시작합니다."),
    ("D03", "별빛회사의 급여일은 매월 25일입니다."),
]
documents = []
for doc_id, content in raw_documents:
    document = Document(page_content=content, metadata={"doc_id": doc_id})
    documents.append(document)
print("문서 수:", len(documents), "타입:", type(documents[0]).__name__)
for document in documents:
    print(document.metadata, document.page_content)

문서 수: 3 타입: Document
{'doc_id': 'D01'} 미르시립도서관의 정기 휴관일은 매주 월요일입니다.
{'doc_id': 'D02'} 해솔식당의 점심 영업은 오전 11시에 시작합니다.
{'doc_id': 'D03'} 별빛회사의 급여일은 매월 25일입니다.


<h3>3. 임베딩 모델과 벡터스토어 만들기</h3><p>각 문서를 뜻을 나타내는 숫자 벡터로 바꾸고, 원문과 벡터를 현재 실행 메모리의 저장소에 함께 넣습니다.</p>

In [4]:
# embedding은 문장을 숫자 벡터로 바꾸는 OpenAI 모델 객체입니다.
# add_documents()는 문서 벡터와 원본 Document를 store에 저장합니다.
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
store = InMemoryVectorStore(embedding=embedding)
stored_ids = store.add_documents(documents)
print("store 타입:", type(store).__name__)
print("저장된 벡터 ID 수:", len(stored_ids))

store 타입: InMemoryVectorStore
저장된 벡터 ID 수: 3


<h3>4. 기본값 k=1로 검색</h3><p>검색 결과는 Document와 유사도 점수를 한 쌍으로 반환합니다. 두 값을 따로 출력해 어떤 문서가 먼저 선택되었는지 확인합니다.</p>

In [5]:
# similarity_search_with_score()는 (Document, 점수) 튜플의 리스트를 반환합니다.
# k=1은 가장 관련 높은 결과 한 건만 요청합니다.
query = "도서관이 쉬는 요일은?"
hits_k1 = store.similarity_search_with_score(query, k=1)
print("반환 타입:", type(hits_k1).__name__, "개수:", len(hits_k1))
for document, score in hits_k1:
    print("doc_id:", document.metadata["doc_id"])
    print("score:", round(score, 3))
    print("content:", document.page_content)

반환 타입: list 개수: 1
doc_id: D01
score: 0.526
content: 미르시립도서관의 정기 휴관일은 매주 월요일입니다.


<h3>5. k만 2로 바꾸어 비교</h3><p>질문·문서·임베딩 모델은 그대로 두고 반환 개수만 늘립니다.</p>

In [6]:
# hits_k2는 같은 query에서 k만 2로 변경한 결과입니다.
# 반복문으로 두 결과의 ID와 점수를 모두 보여 줍니다.
hits_k2 = store.similarity_search_with_score(query, k=2)
print("k=1 개수:", len(hits_k1))
print("k=2 개수:", len(hits_k2))
for document, score in hits_k2:
    print(document.metadata["doc_id"], round(score, 3))
assert hits_k1[0][0].metadata["doc_id"] == "D01"
assert len(hits_k2) == 2

k=1 개수: 1
k=2 개수: 2
D01 0.526
D03 0.325


<h3>정리</h3><p>뜻이 가까운 문서를 찾는 검색에서 D01이 첫 번째로 나왔습니다. <code>k</code>만 1에서 2로 늘리자 D01은 그대로 있고 문서 한 건이 더 보였습니다.</p>

<p><strong>원인</strong> — 도서관 휴관일과 회사 급여일을 함께 묻지만 <code>k=1</code>은 문서 한 건만 반환하므로 두 답의 근거를 모두 가져올 수 없습니다.</p>

In [7]:
# combined_query는 서로 다른 두 문서의 정보가 필요한 질문입니다.
# k=1에서 빠진 근거를 ID와 본문으로 직접 확인합니다.
combined_query = "도서관 휴관일과 회사 급여일은 언제인가요?"
before_hits = store.similarity_search(combined_query, k=1)
before_ids = [document.metadata["doc_id"] for document in before_hits]
print("Before ID:", before_ids)
for document in before_hits:
    print("Before 본문:", document.page_content)
try:
    assert {"D01", "D03"}.issubset(before_ids)
except AssertionError:
    print("예상한 실패: 질문에 필요한 문서 두 건을 모두 가져오지 못했습니다.")
else:
    raise AssertionError("k=1 검색 누락이 재현되어야 합니다.")

Before ID: ['D01']
Before 본문: 미르시립도서관의 정기 휴관일은 매주 월요일입니다.
예상한 실패: 질문에 필요한 문서 두 건을 모두 가져오지 못했습니다.


<p><strong>수정</strong> — 질문과 문서는 유지하고 <code>k</code>만 1에서 2로 늘려 두 답의 근거 문서를 함께 검색합니다.</p>

In [8]:
# fixed_hits는 같은 combined_query에서 k만 2로 바꾼 결과입니다.
# D01과 D03의 ID와 본문이 모두 들어왔는지 검사합니다.
fixed_hits = store.similarity_search(combined_query, k=2)
fixed_ids = [document.metadata["doc_id"] for document in fixed_hits]
print("After ID:", fixed_ids)
for document in fixed_hits:
    print("After 본문:", document.page_content)
assert {"D01", "D03"}.issubset(fixed_ids)

After ID: ['D01', 'D03']
After 본문: 미르시립도서관의 정기 휴관일은 매주 월요일입니다.
After 본문: 별빛회사의 급여일은 매월 25일입니다.
